[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-1/router.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239412-lesson-5-router)

# 路由器

## 回顾

我们构建了一个使用 `消息` 作为状态并绑定工具的聊天模型的图。

我们看到图可以：

* 返回工具调用
* 返回自然语言响应

## 目标

我们可以将其视为路由器，聊天模型根据用户输入在直接响应或工具调用之间进行路由。

这是一个简单的 agent 示例，LLM 通过调用工具或直接响应来指导控制流。

![Screenshot 2024-08-21 at 9.24.09 AM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbac6543c3d4df239a4ed1_router1.png)

让我们扩展我们的图以处理任一输出！

为此，我们可以使用两个想法：

(1) 添加一个将调用我们工具的节点。

(2) 添加一个条件边，它将查看聊天模型输出，并路由到我们的工具调用节点，或者如果没有执行工具调用则简单地结束。

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langchain_openai langchain_core langgraph langgraph-prebuilt

In [ ]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")

In [ ]:
from langchain_openai import ChatOpenAI

def multiply(a: int, b: int) -> int:
    """将 a 和 b 相乘。

    Args:
        a: 第一个整数
        b: 第二个整数
    """
    return a * b

llm = ChatOpenAI(model="gpt-4o")
llm_with_tools = llm.bind_tools([multiply])

我们使用 [内置的 `ToolNode`](https://langchain-ai.github.io/langgraph/reference/prebuilt/?h=tools+condition#toolnode)，并简单地传递我们的工具列表来初始化它。

我们使用 [内置的 `tools_condition`](https://langchain-ai.github.io/langgraph/reference/prebuilt/?h=tools+condition#tools_condition) 作为我们的条件边。

In [ ]:
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition

# 节点
def tool_calling_llm(state: MessagesState):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

# 构建图
builder = StateGraph(MessagesState)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode([multiply]))
builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges(
    "tool_calling_llm",
    # 如果 assistant 的最新消息（结果）是工具调用 -> tools_condition 路由到 tools
    # 如果 assistant 的最新消息（结果）不是工具调用 -> tools_condition 路由到 END
    tools_condition,
)
builder.add_edge("tools", END)
graph = builder.compile()

# 查看
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
from langchain_core.messages import HumanMessage
messages = [HumanMessage(content="你好，2 乘以 2 是多少？")]
messages = graph.invoke({"messages": messages})
for m in messages['messages']:
    m.pretty_print()

现在，我们可以看到图运行了工具！

它用 `ToolMessage` 响应。

## LangGraph Studio

**⚠️ 免责声明**

自这些视频拍摄以来，我们已经更新了 Studio，使其可以在本地运行并在浏览器中打开。这现在是运行 Studio 的首选方式（而不是像视频中显示的使用桌面应用程序）。请参阅 [此处](https://langchain-ai.github.io/langgraph/concepts/langgraph_studio/#local-development-server) 关于本地开发服务器的文档和 [此处](https://langchain-ai.github.io/langgraph/how-tos/local-studio/#run-the-development-server) 的更多信息。要启动本地开发服务器，请在此模块的 `/studio` 目录中的终端中运行以下命令：

```
langgraph dev
```

您应该看到以下输出：
```
- 🚀 API: http://127.0.0.1:2024
- 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs
```

打开浏览器并导航到 Studio UI：`https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024`。
在 Studio 中加载 `router`，它使用在 `module-1/studio/langgraph.json` 中设置的 `module-1/studio/router.py`。

In [ ]:
if 'google.colab' in str(get_ipython()):
    raise Exception("抱歉，LangGraph Studio 目前不支持 Google Colab")